### Análise Exploratória de Dados (EDA) inicial do projeto. 

##### Aqui, vamos buscar entender quais são os pontos de atenção para a realização posterior do pré-processamento.

In [3]:
import pandas as pd

In [4]:
orderItems = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
orderPayments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
orderReviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orderDataset = pd.read_csv("../data/raw/olist_orders_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
nameTranslation = pd.read_csv("../data/raw/product_category_name_translation.csv")

### Iniciando a análise exploratória dos dados (EDA)

In [5]:
dfs = {
    "items": orderItems,
    "payments": orderPayments,
    "reviews": orderReviews,
    "orders": orderDataset,
    "customers": customers,
    "geolocation": geolocation,
    "products": products,
    "sellers": sellers,
    "nameTranslation": nameTranslation
}

for nome, df in dfs.items():
    print(nome, df.shape)

items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
orders (99441, 8)
customers (99441, 5)
geolocation (1000163, 5)
products (32951, 9)
sellers (3095, 4)
nameTranslation (71, 2)


### Como o atraso se distribui?

In [6]:
orderDataset.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


### Aqui, temos as informações de:

* Status da entrega
* A data que o pedido foi realizado
* A data que o pedido foi aprovado
* A data que o vendedor despachou o pacote
* A data que o cliente recebeu de fato o produto
* A data estimada em que o pedido chegaria até o cliente

Naturalmente, se a data que o cliente recebeu for após a data estimada, isso se enquadra como um atraso. Daqui, conseguimos tirar uma label de "entregue atrasado" e "entregue no prazo".

As datas em que o pedido foi realizado, aprovado e despachado podem fazer parte de uma análise complementar.

Os pedidos taggados como "delivered" possuem as informações de data de postagem, data estimada de entrega e data em que a entrega aconteceu de fato. Mas, temos outras possibilidades: "invoiced", ou seja, que já foram pagos mas o vendedor ainda não enviou, e também os "shipped", que foram enviados mas ainda não chegaram.

In [7]:
orderDataset["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Apesar disso, acredito que, pela distribuição dessa coluna, pode ser mais "confiável" utilizar só onde order_status == delivered, já que, para os outros casos, existem muitos valores faltantes que só trariam mais dor de cabeça, e representam somente cerca de 3% do total (delivered, portanto, representa 97% do dataset).

In [39]:
orderDataset[orderDataset["order_status"] == "unavailable"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
266,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,2017-11-16 15:09:28,2017-11-16 15:26:57,NaN,NaN,2017-12-05 00:00:00
586,c272bcd21c287498b4883c7512019702,9582c5bbecc65eb568e2c1d839b5cba1,unavailable,2018-01-31 11:31:37,2018-01-31 14:23:50,NaN,NaN,2018-02-16 00:00:00
687,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,unavailable,2017-08-14 17:38:02,2017-08-17 00:15:18,NaN,NaN,2017-09-05 00:00:00
737,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,unavailable,2018-01-08 19:39:03,2018-01-09 07:26:08,NaN,NaN,2018-02-06 00:00:00
1160,2f634e2cebf8c0283e7ef0989f77d217,7353b0fb8e8d9675e3a704c60ca44ebe,unavailable,2017-09-27 20:55:33,2017-09-28 01:32:50,NaN,NaN,2017-10-27 00:00:00


In [30]:
entregues = orderDataset[orderDataset["order_status"] == "delivered"]

entregues["order_delivered_customer_date"] = pd.to_datetime(entregues["order_delivered_customer_date"])
entregues["order_estimated_delivery_date"] = pd.to_datetime(entregues["order_estimated_delivery_date"])

atrasado = entregues[entregues["order_delivered_customer_date"] > entregues["order_estimated_delivery_date"]]

print(f"Proporção de atrasados: {atrasado.shape[0] / entregues.shape[0]:.2%}")
print(f"Proporção de no prazo: {1 - atrasado.shape[0] / entregues.shape[0]:.2%}")

Proporção de atrasados: 8.11%
Proporção de no prazo: 91.89%


Em tese isso aqui faz parte do pré-processamento, mas organizamos depois.

In [31]:
entregues["order_delivered_customer_date"].dtype

dtype('<M8[us]')

In [60]:
atrasado = entregues[entregues["dias_atrasados"] > 0]
atrasado["dias_atrasados"].describe()

count    6534.000000
mean       10.620141
std        14.644955
min         1.000000
25%         3.000000
50%         7.000000
75%        13.000000
max       188.000000
Name: dias_atrasados, dtype: float64

orderItems se relaciona com orderDataset (e entregues e atrasado) com "order_id", então conseguimos puxar o preço de venda, além do frete

In [33]:
orderItems.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
